# Descargador de fuentes de arXiv a partir de un listado de identificadores (v4)
Este cuaderno descarga los *tarballs* de código fuente (`https://arxiv.org/src/<id>`) de los artículos indicados en el archivo **`semilla.txt`**. Antes de descargar verifica que exista la carpeta de la semilla (la crea si no existe), crea automáticamente la carpeta de salida **`papers-origen/`** y guarda allí solo las fuentes que sean tar.gz válidos.

**Cambios respecto a la v2**
- La carpeta `papers-origen/` se crea automáticamente al cargar los parámetros y se vuelve a asegurar antes de descargar y antes de escribir los CSV, de modo que exista aunque las celdas se ejecuten fuera de orden.
- Se eliminó la sección de empaquetado en ZIP y división en trozos.
- `index.csv` contiene únicamente las columnas `arxiv_id,primary_category`.

Respeta las pautas de arXiv: una petición a la vez, ~3 s entre peticiones, User-Agent descriptivo y reintentos con espera creciente ante 429/503.

## 1. Parámetros
- `SEED_DIR` / `SEED_FILENAME`: carpeta y nombre del listado de identificadores (`semilla.txt`). Si la carpeta no existe, se crea.
- `OUT_DIR`: carpeta `papers-origen` donde se guardan las descargas. Se crea automáticamente al ejecutar esta celda.
- `N_ARTICLES`: `None` descarga todos los IDs de la semilla; un entero limita el número de tarballs válidos.
- `FETCH_METADATA`: consulta la API para obtener la categoría principal (columna `primary_category` de `index.csv`).
- `APPLY_CATEGORY_FILTER`: si es `True`, solo se descargan los IDs que pasen el filtro (`CATEGORY_PREFIX`, `ALLOWED_SUBCATS`, `FILTER_ON`).

**Formato de `semilla.txt`**: un identificador por línea (también se aceptan varios por línea separados por comas o espacios). Valen formas como `2605.01234`, `2605.01234v2`, `arXiv:2605.01234`, `https://arxiv.org/abs/2605.01234` o IDs antiguos como `hep-th/9901001`. Las líneas vacías y lo que vaya después de `#` se ignoran; los duplicados se eliminan.

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')


Mounted at /content/drive


In [7]:
import os

BASE_DIR = "/home/fgamez/drive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-balanceada-2000/tanda-02-1200"                  # carpeta base de trabajo
SEED_DIR = f"{BASE_DIR}"         # carpeta donde se busca la semilla (se crea si no existe)
SEED_FILENAME = f"{BASE_DIR}/lista-articulos-destino.txt"            # listado de identificadores de arXiv
OUT_DIR = f"{BASE_DIR}/papers-destino"    # carpeta de descargas (se crea automáticamente)
SKIPPED_CSV = f"{OUT_DIR}/skipped.csv"   # IDs omitidos y motivo
N_ARTICLES = None           # None = todos los IDs de la semilla; un entero limita los tarballs válidos
FETCH_METADATA = True       # pedir la categoría principal a la API para index.csv
APPLY_CATEGORY_FILTER = False   # True = descargar solo los IDs que pasen el filtro de categoría
CATEGORY_PREFIX = "cs."     # usado solo si APPLY_CATEGORY_FILTER = True
ALLOWED_SUBCATS = None      # p.ej. ["cs.LG", "cs.CL", "cs.CV"]; None = todas las cs.*
FILTER_ON = "primary"       # "primary" o "any"
SKIP_EXISTING = True        # no volver a descargar tarballs válidos ya presentes en OUT_DIR
DELAY = 3.0                 # segundos entre peticiones
MAX_RETRIES = 6
USER_AGENT = "Mozilla 5.0"

# Crear la carpeta de salida papers-origen/ de forma automática
_existia = os.path.isdir(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Carpeta de salida {'encontrada' if _existia else 'creada'}: {OUT_DIR}")

Carpeta de salida encontrada: /home/fgamez/drive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-balanceada-2000/tanda-02-1200/papers-destino


## 2. Utilidades HTTP, API e identificadores
`http_get` añade el User-Agent, espera `DELAY` segundos tras cada petición y reintenta con espera exponencial ante 429/5xx o errores de red. `parse_feed` extrae ID, título, categoría principal y todas las categorías de cada entrada del feed Atom (ignora las entradas de error de la API). `extract_ids` reconoce identificadores nuevos (`AAMM.NNNNN`) y antiguos (`archivo/NNNNNNN`) dentro de una línea y les quita la versión. `safe_name` reemplaza `/` para poder usar los IDs antiguos como nombre de archivo. `ensure_out_dir` garantiza que `papers-origen/` exista antes de escribir en ella.

In [8]:
import os, re, csv, time, shutil, subprocess, urllib.request, urllib.error, urllib.parse
import xml.etree.ElementTree as ET
from collections import Counter
NS = {'a': 'http://www.w3.org/2005/Atom', 'x': 'http://arxiv.org/schemas/atom',
      'os': 'http://a9.com/-/spec/opensearch/1.1/'}
assert shutil.which('file'), "Instala la utilidad 'file' (apt-get install file)"

def ensure_out_dir():
    """Crea papers-origen/ si no existe (idempotente)."""
    os.makedirs(OUT_DIR, exist_ok=True)
    return OUT_DIR

def http_get(url, timeout=180):
    wait = DELAY
    for attempt in range(MAX_RETRIES):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': USER_AGENT})
            data = urllib.request.urlopen(req, timeout=timeout).read()
            time.sleep(DELAY)
            return data
        except urllib.error.HTTPError as e:
            if e.code not in (429, 500, 502, 503, 504) or attempt == MAX_RETRIES - 1:
                time.sleep(DELAY); raise
        except (urllib.error.URLError, TimeoutError):
            if attempt == MAX_RETRIES - 1: raise
        wait = min(wait * 2, 120)
        print(f"  reintento {attempt + 1} en {wait:.0f} s ...")
        time.sleep(wait)

def parse_feed(xml_bytes):
    root = ET.fromstring(xml_bytes)
    total = int(root.find('os:totalResults', NS).text or 0)
    entries = []
    for e in root.findall('a:entry', NS):
        raw_id = e.find('a:id', NS).text or ''
        if '/abs/' not in raw_id:        # entradas de error de la API
            continue
        aid = re.sub(r'v\d+$', '', raw_id.split('/abs/', 1)[1])
        title = " ".join((e.find('a:title', NS).text or '').split())
        pc = e.find('x:primary_category', NS)
        cats = [c.get('term') for c in e.findall('a:category', NS)]
        entries.append({'id': aid, 'title': title,
                        'primary': pc.get('term') if pc is not None else '', 'cats': cats})
    return total, entries

def meta_by_ids(ids):
    url = "https://export.arxiv.org/api/query?" + urllib.parse.urlencode(
        {'id_list': ",".join(ids), 'max_results': len(ids)})
    return parse_feed(http_get(url))[1]

def cat_ok(entry):
    def ok(c):
        return c.startswith(CATEGORY_PREFIX) and (ALLOWED_SUBCATS is None or c in ALLOWED_SUBCATS)
    return ok(entry['primary']) if FILTER_ON == "primary" else any(ok(c) for c in entry['cats'])

# IDs nuevos (2605.01234[v2]) y antiguos (hep-th/9901001[v1], math.AG/0101001)
ID_RE = re.compile(r'(?<![\w.])(\d{4}\.\d{4,5}|[a-z][a-z\-]*(?:\.[A-Z]{2})?/\d{7})(?:v\d+)?(?!\d)')
def extract_ids(text):
    return ID_RE.findall(text)

safe_name = lambda aid: aid.replace('/', '_')

def check_tar(path):
    ft = subprocess.run(['file', '-b', path], capture_output=True, text=True).stdout.strip()
    ok = ft.startswith('gzip') and subprocess.run(['tar', '-tzf', path], capture_output=True).returncode == 0
    return ft, ok

## 3. Verificar la carpeta de la semilla y leer `semilla.txt`
Si `SEED_DIR` no existe, se crea. Si falta `semilla.txt`, la celda se detiene con un mensaje que indica dónde colocarlo; después de copiarlo basta con volver a ejecutar esta celda.

In [9]:
SEED_FILE = os.path.join(SEED_DIR, SEED_FILENAME)
if not os.path.isdir(SEED_DIR):
    os.makedirs(SEED_DIR, exist_ok=True)
    print(f"Carpeta de semilla creada: {SEED_DIR}")
else:
    print(f"Carpeta de semilla encontrada: {SEED_DIR}")

if not os.path.isfile(SEED_FILE):
    raise FileNotFoundError(
        f"No se encontró {SEED_FILE}. Coloca allí '{SEED_FILENAME}' con un identificador de arXiv "
        "por línea y vuelve a ejecutar esta celda.")

seed_ids, invalid_lines = [], []
with open(SEED_FILE, encoding='utf-8-sig') as f:
    for lineno, line in enumerate(f, 1):
        text = line.split('#', 1)[0].strip()
        if not text:
            continue
        found = extract_ids(text)
        if found:
            seed_ids.extend(found)
        else:
            invalid_lines.append((lineno, text))

n_raw = len(seed_ids)
seed_ids = list(dict.fromkeys(seed_ids))     # sin duplicados, conservando el orden
print(f"IDs leídos: {n_raw} | únicos: {len(seed_ids)} | líneas no reconocidas: {len(invalid_lines)}")
for lineno, text in invalid_lines[:20]:
    print(f"  línea {lineno} ignorada: {text[:80]}")
if not seed_ids:
    raise ValueError(f"{SEED_FILE} no contiene identificadores de arXiv reconocibles.")
print("Primeros IDs:", seed_ids[:5])

Carpeta de semilla encontrada: /home/fgamez/drive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-balanceada-2000/tanda-02-1200
IDs leídos: 4924 | únicos: 1881 | líneas no reconocidas: 0
Primeros IDs: ['0912.0926', '1005.3450', '1111.3328', '0807.3286', '1206.5115']


## 4. Verificar la carpeta de salida `papers-origen/`
Se vuelve a asegurar que exista (se crea si falta) y se elimina cualquier archivo temporal que haya quedado de una ejecución interrumpida.

In [10]:
ensure_out_dir()
assert os.path.isdir(OUT_DIR), f"No se pudo crear la carpeta de salida {OUT_DIR}"
print(f"Carpeta de salida lista: {OUT_DIR}")
TMP_PATH = os.path.join(OUT_DIR, '.tmp')
if os.path.exists(TMP_PATH):
    os.remove(TMP_PATH)

Carpeta de salida lista: /home/fgamez/drive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-balanceada-2000/tanda-02-1200/papers-destino


## 5. Metadatos y filtro opcional por categoría
Si `FETCH_METADATA` es `True`, se piden los metadatos a la API en lotes de 50 (`id_list`) para tener la categoría principal en el índice. Si `APPLY_CATEGORY_FILTER` es `True`, los IDs que no pasan el filtro (o de los que no se obtuvieron metadatos) se registran como omitidos y no se descargan.

In [11]:
meta, skipped = {}, []
if FETCH_METADATA or APPLY_CATEGORY_FILTER:
    for i in range(0, len(seed_ids), 50):
        batch = seed_ids[i:i + 50]
        try:
            for e in meta_by_ids(batch):
                meta[e['id']] = e
        except Exception as ex:
            print(f"  lote {i // 50 + 1}: falló la consulta de metadatos ({ex})")
        print(f"  metadatos: {min(i + 50, len(seed_ids))}/{len(seed_ids)} IDs consultados")
    missing = [a for a in seed_ids if a not in meta]
    print(f"Metadatos obtenidos: {len(meta)} | sin metadatos: {len(missing)}", missing[:10])
    print("Categorías principales más comunes:", Counter(e['primary'] for e in meta.values()).most_common(10))

queue = []
for aid in seed_ids:
    if APPLY_CATEGORY_FILTER:
        e = meta.get(aid)
        if e is None:
            skipped.append((aid, 'sin metadatos (no se pudo aplicar el filtro de categoría)')); continue
        if not cat_ok(e):
            skipped.append((aid, f"fuera de categoría ({e['primary']})")); continue
    queue.append(aid)
print(f"IDs en cola de descarga: {len(queue)} | omitidos por filtro: {len(skipped)}")

  metadatos: 50/1881 IDs consultados
  metadatos: 100/1881 IDs consultados
  metadatos: 150/1881 IDs consultados
  metadatos: 200/1881 IDs consultados
  metadatos: 250/1881 IDs consultados
  metadatos: 300/1881 IDs consultados
  metadatos: 350/1881 IDs consultados
  metadatos: 400/1881 IDs consultados
  metadatos: 450/1881 IDs consultados
  metadatos: 500/1881 IDs consultados
  metadatos: 550/1881 IDs consultados
  metadatos: 600/1881 IDs consultados
  metadatos: 650/1881 IDs consultados
  metadatos: 700/1881 IDs consultados
  metadatos: 750/1881 IDs consultados
  metadatos: 800/1881 IDs consultados
  metadatos: 850/1881 IDs consultados
  metadatos: 900/1881 IDs consultados
  metadatos: 950/1881 IDs consultados
  metadatos: 1000/1881 IDs consultados
  metadatos: 1050/1881 IDs consultados
  metadatos: 1100/1881 IDs consultados
  metadatos: 1150/1881 IDs consultados
  metadatos: 1200/1881 IDs consultados
  metadatos: 1250/1881 IDs consultados
  metadatos: 1300/1881 IDs consultados
  meta

## 6. Descarga y validación de tarballs en `papers-origen/`
Para cada ID de la cola se descarga `/src/<id>` (última versión), se comprueba con `file` que sea gzip y con `tar -tzf` que sea un tar válido, y se guarda como `papers-origen/<id>.tar.gz`. Los que son solo PDF o un único archivo comprimido se registran como omitidos. Con `SKIP_EXISTING`, los tarballs válidos ya presentes se reutilizan sin volver a descargarlos.

In [12]:
ensure_out_dir()
TMP_PATH = os.path.join(OUT_DIR, '.tmp')
rows = []   # (arxiv_id, primary_category)
primary_of = lambda a: meta[a]['primary'] if a in meta else ''
limit = N_ARTICLES or len(queue)
for aid in queue:
    if len(rows) >= limit: break
    dst = os.path.join(OUT_DIR, f"{safe_name(aid)}.tar.gz")
    dst_gz = os.path.join(OUT_DIR, f"{safe_name(aid)}.gz")   # >>> CAMBIO 1: nueva línea, destino para .gz que no son tar
    if SKIP_EXISTING and os.path.exists(dst) and check_tar(dst)[1]:
        rows.append((aid, primary_of(aid)))
        print(f"{aid} ya existe, se conserva ({len(rows)}/{limit})"); continue
    # >>> CAMBIO 2: nuevo bloque, reutilizar .gz ya descargados (se valida con gzip -t)
    if SKIP_EXISTING and os.path.exists(dst_gz) and subprocess.run(['gzip', '-t', dst_gz], capture_output=True).returncode == 0:
        rows.append((aid, primary_of(aid)))
        print(f"{aid} (.gz) ya existe, se conserva ({len(rows)}/{limit})"); continue
    # <<< fin CAMBIO 2
    try:
        with open(TMP_PATH, 'wb') as f: f.write(http_get(f"https://arxiv.org/src/{aid}", 300))
    except Exception as ex:
        skipped.append((aid, f'download error: {ex}')); print(aid, 'error:', ex)
        if os.path.exists(TMP_PATH): os.remove(TMP_PATH)
        continue
    ft, ok = check_tar(TMP_PATH)
    if ok:
        os.replace(TMP_PATH, dst)
        rows.append((aid, primary_of(aid)))
        print(f"{aid} [{primary_of(aid)}] OK ({len(rows)}/{limit})")
    # >>> CAMBIO 3: nuevo bloque elif, un gzip que no es tar se acepta y se guarda como .gz
    elif ft.startswith('gzip'):
        os.replace(TMP_PATH, dst_gz)
        rows.append((aid, primary_of(aid)))
        print(f"{aid} [{primary_of(aid)}] OK .gz ({len(rows)}/{limit})")
    # <<< fin CAMBIO 3
    else:
        why = 'PDF only' if 'PDF' in ft else ft[:60]   # >>> CAMBIO 4: reemplaza la línea "why = 'single gzipped file (not tar)' if ..."
        skipped.append((aid, why)); os.remove(TMP_PATH); print(aid, 'omitido:', why)
print("Hecho:", len(rows), "archivos en", OUT_DIR, ";", len(skipped), "omitidos")   # >>> CAMBIO 5 (opcional): "tarballs" -> "archivos"

0912.0926 [cs.OS] OK (1/1881)
1005.3450 [cs.OS] OK (2/1881)
1111.3328 [quant-ph] OK (3/1881)
0807.3286 [quant-ph] OK (4/1881)
1206.5115 [quant-ph] OK .gz (5/1881)
0911.3427 [quant-ph] OK (6/1881)
1111.6054 [quant-ph] OK (7/1881)
0709.1516 [math.ST] OK .gz (8/1881)
1203.4740 [quant-ph] OK (9/1881)
1004.5127 [quant-ph] OK (10/1881)
1105.1827 [cs.DC] OK (11/1881)
1304.4134 [cs.SC] OK .gz (12/1881)
1408.2776 [cs.SC] OK (13/1881)
1603.04285 [cs.SC] OK (14/1881)
1509.08324 [hep-ph] OK (15/1881)
1307.7887 [cs.SC] OK .gz (16/1881)
1606.01131 [math.NT] OK .gz (17/1881)
1307.1482 omitido: PDF only
1706.06927 [cs.RO] OK (18/1881)
1312.4328 [cs.PL] OK (19/1881)
1609.05518 [cs.AI] OK (20/1881)
1801.04016 [cs.LG] OK (21/1881)
1511.03774 [cs.LG] OK (22/1881)
1707.02286 [cs.AI] OK (23/1881)
1611.01142 [cs.LG] OK (24/1881)
1707.06347 [cs.LG] OK (25/1881)
1602.01783 [cs.LG] OK (26/1881)
1412.6980 [cs.LG] OK (27/1881)
1506.02438 [cs.LG] OK (28/1881)
1302.7008 [cs.GT] OK (29/1881)
1804.07643 [cs.RO] OK (3

## 7. Escribir `index.csv` y `skipped.csv` en `papers-origen/`
`index.csv` contiene únicamente dos columnas: `arxiv_id,primary_category`.

In [13]:
ensure_out_dir()
rows.sort()
with open(os.path.join(OUT_DIR, 'index.csv'), 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['arxiv_id', 'primary_category']); w.writerows(rows)
with open(SKIPPED_CSV, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['arxiv_id', 'reason']); w.writerows(skipped)
print(f"index.csv: {len(rows)} filas | skipped.csv: {len(skipped)} filas | carpeta: {OUT_DIR}")

index.csv: 1804 filas | skipped.csv: 77 filas | carpeta: /home/fgamez/drive/05-EDUCATION/2025-MAIA/2026-PROYECTO-GRADO-MAIA/arxiv-dataset-semilla-balanceada-2000/tanda-02-1200/papers-destino
